<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/dlai-generative-ai-with-large-language-models/fine_tune_model_to_detoxify_summaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!hf download palarunava/dlai-genaillm-flan-t5-dialogsum-lora --local-dir ./peft-dialogue-summary-checkpoint

Fetching 6 files: 100% 6/6 [00:02<00:00,  2.68it/s]
Download complete: : 16.6MB [00:02, 9.28MB/s]              ✓ Downloaded
  path: /content/peft-dialogue-summary-checkpoint
Download complete: : 16.6MB [00:02, 6.87MB/s]


In [3]:
# from getpass import getpass
# import os

# user = "palarunava"
# access_token = getpass('Enter your GitHub PAT: ')
# repo = "learning-lab"

# # Build the URL and clone
# !git clone https://{access_token}@github.com/{user}/{repo}.git

# # Optional: Clear the password variable from memory
# password = ""

In [4]:
# !pip install -U \
#     torch==2.10.0 \
#     evaluate \
#     rouge_score \
#     peft \
#     torchao \
#     transformers \
#     datasets \
#     pandas==2.2.2 \
#     numpy==2.0 \
#     accelerate \
#     trl --quiet

# !pip install -U \
#     torch==2.5.1 \
#     datasets==2.17.0 \
#     transformers==4.38.2 \
#     evaluate==0.4.0 \
#     peft==0.3.0 \
#     trl==0.4.2 --quiet

# !pip install \
#     trl \
#     torchao==0.16.0 \
#     evaluate --quiet

# !pip install -U \
#     torch==2.5.1 \
#     pandas==2.3.3 \
#     numpy==1.26.0 \
#     datasets==2.17.0 \
#     transformers==4.38.2 \
#     accelerate==0.28.0 \
#     evaluate==0.4.0 \
#     rouge_score==0.1.2 \
#     peft==0.3.0 \
#     trl==0.4.2 \
#     torchao==0.16.0 --force-reinstall --no-deps --quiet

!pip install trl==0.4.7 transformers==4.38.2 peft==0.3.0 evaluate sentence-transformers==2.7.0 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 37.1 MB/s eta 0:00:00


In [5]:
# Replace: from trl.core import LengthSampler
# With this:
# import numpy as np

# class LengthSampler:
#     def __init__(self, min_value, max_value):
#         self.min_value = min_value
#         self.max_value = max_value

#     def __call__(self):
#         return np.random.randint(self.min_value, self.max_value)

In [6]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import warnings
warnings.filterwarnings('ignore')

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, GenerationConfig
from datasets import load_dataset
from peft import PeftModel, PeftConfig, LoraConfig, TaskType

# trl: Transformer Reinforcement Learning library
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead
from trl import create_reference_model
from trl.core import LengthSampler

import torch
import evaluate

import numpy as np
import pandas as pd

# tqdm library makes the loops show a smart progress meter.
from tqdm import tqdm
tqdm.pandas()

import json

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [7]:
model_name = 'google/flan-t5-base'
huggingface_dataset_name = 'knkarthick/dialogsum'

dataset_original = load_dataset(huggingface_dataset_name)

dataset_original

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [8]:
# Set the device to GPU or CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
def build_dataset(model_name,
                  dataset_name,
                  input_min_text_length,
                  input_max_text_length):
    """
    Preprocess the dataset and split it into train and test sets.

    Parameters:
    - model_name (str): Tokenizer model name.
    - dataset_name (str): Name of the dataset to load.
    - input_min_text_length (int): Minimum length of the dialogues.
    - input_max_text_length (int): Maximum length of the dialogues.

    Returns:
    - dataset_splits (datasets.dataset_dict.DatasetDict): Preprocessed dataset containing train and test parts.
    """

    # Load dataset (only "train" part will be enough for this lab)
    dataset = load_dataset(dataset_name, split='train')

    # Filter the dialogues of length between input_min_text_length and input_max_text_length characters.
    dataset = dataset.filter(lambda x: len(x['dialogue']) > input_min_text_length and len(x['dialogue']) <= input_max_text_length, batched=False)

    # Prepare tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)

    def tokenize(sample):

        # Wrap each dialogue with the instruction.
        prompt = f"""
Summarize the following conversation.

{sample['dialogue']}

Summary:
"""
        sample['input_ids'] = tokenizer.encode(prompt)

        # This must be called "query", which is a requirement of our PPO library.
        sample['query'] = tokenizer.decode(sample['input_ids'])
        return sample

    # Tokenize each dialogue.
    dataset = dataset.map(tokenize, batched=False)
    dataset.set_format(type='torch')

    # Split the dataset into train and test parts.
    dataset_splits = dataset.train_test_split(test_size=0.2, shuffle=False, seed=42)

    return dataset_splits

dataset = build_dataset(model_name=model_name,
                        dataset_name=huggingface_dataset_name,
                        input_min_text_length=200,
                        input_max_text_length=1000)
print(dataset)

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10022 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'query'],
        num_rows: 8017
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'query'],
        num_rows: 2005
    })
})


In [10]:
# !cp -r \
#   '/content/learning-lab/DeepLearning.AI/Generative AI with Large Language Models/Week 2 - Fine-tuning and evaluating large language models/Practice Lab/models/peft-dialogue-summary-checkpoint' \
#   .

In [11]:
!ls -al ./peft-dialogue-summary-checkpoint/

total 16272
drwxr-xr-x 3 root root     4096 May  1 03:41 .
drwxr-xr-x 1 root root     4096 May  1 03:41 ..
-rw-r--r-- 1 root root      334 May  1 03:41 adapter_config.json
-rw-r--r-- 1 root root 14208525 May  1 03:41 adapter_model.bin
drwxr-xr-x 3 root root     4096 May  1 03:41 .cache
-rw-r--r-- 1 root root     1519 May  1 03:41 .gitattributes
-rw-r--r-- 1 root root     2201 May  1 03:41 special_tokens_map.json
-rw-r--r-- 1 root root     2496 May  1 03:41 tokenizer_config.json
-rw-r--r-- 1 root root  2422164 May  1 03:41 tokenizer.json


In [12]:
!ls -alh ./peft-dialogue-summary-checkpoint/adapter_model.bin

-rw-r--r-- 1 root root 14M May  1 03:41 ./peft-dialogue-summary-checkpoint/adapter_model.bin


In [13]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"\ntrainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

In [14]:
lora_config = LoraConfig(
    r=32, # Rank
    lora_alpha=32,
    target_modules=['q', 'v'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM # FLAN-T5
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

peft_model = PeftModel.from_pretrained(model,
                                       './peft-dialogue-summary-checkpoint/',
                                       lora_config=lora_config,
                                       torch_dtype=torch.bfloat16,
                                       device_map=device,
                                       is_trainable=True)
print(f'PEFT model parameters to be updated:\n{print_number_of_trainable_model_parameters(peft_model)}\n')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

PEFT model parameters to be updated:

trainable model parameters: 3538944
all model parameters: 251116800
percentage of trainable model parameters: 1.41%



In [15]:
ppo_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(peft_model,
                                                               torch_dtype=torch.bfloat16,
                                                               is_trainable=True)
print(f'PPO model parameters to be updated (ValueHead + 769 params):\n{print_number_of_trainable_model_parameters(ppo_model)}\n')
print(ppo_model.v_head)

PPO model parameters to be updated (ValueHead + 769 params):

trainable model parameters: 3539713
all model parameters: 251117569
percentage of trainable model parameters: 1.41%

ValueHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (summary): Linear(in_features=768, out_features=1, bias=True)
  (flatten): Flatten(start_dim=1, end_dim=-1)
)


In [16]:
ref_model = create_reference_model(ppo_model)

print(f'Reference model parameters to be updated:\n{print_number_of_trainable_model_parameters(ref_model)}\n')

Reference model parameters to be updated:

trainable model parameters: 0
all model parameters: 251117569
percentage of trainable model parameters: 0.00%



In [17]:
toxicity_model_name = 'facebook/roberta-hate-speech-dynabench-r4-target'
toxicity_tokenizer = AutoTokenizer.from_pretrained(toxicity_model_name, device_map=device)
toxicity_model = AutoModelForSequenceClassification.from_pretrained(toxicity_model_name, device_map=device)
print(toxicity_model.config.id2label)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

{0: 'nothate', 1: 'hate'}


In [18]:
non_toxic_text = "#Person 1# tells Tommy that he didn't like the movie."

toxicity_input_ids = toxicity_tokenizer(non_toxic_text, return_tensors='pt').input_ids.to(device)

logits = toxicity_model(input_ids=toxicity_input_ids).logits
print(f'logits [not hate, hate]: {logits.tolist()[0]}')

# Print the probabilities for [not hate, hate]
probabilities = logits.softmax(dim=-1).tolist()[0]
print(f'probabilities [not hate, hate]: {probabilities}')

# Get the logits for "not hate" - this is the reward!
not_hate_index = 0
nothate_reward = (logits[:, not_hate_index]).tolist()
print(f'reward (high): {nothate_reward}')

logits [not hate, hate]: [3.114103078842163, -2.4896199703216553]
probabilities [not hate, hate]: [0.9963293671607971, 0.0036705988459289074]
reward (high): [3.114103078842163]


In [19]:
toxic_text = "#Person 1# tells Tommy that the movie was terrible, dumb and stupid."

toxicity_input_ids = toxicity_tokenizer(toxic_text, return_tensors='pt').input_ids.to(device)

logits = toxicity_model(input_ids=toxicity_input_ids).logits
print(f'logits [not hate, hate]: {logits.tolist()[0]}')

# Print the probabilities for [not hate, hate]
probabilities = logits.softmax(dim=-1).tolist()[0]
print(f'probabilities [not hate, hate]: {probabilities}')

# Get the logits for "not hate" - this is the reward!
nothate_reward = (logits[:, not_hate_index]).tolist()
print(f'reward (low): {nothate_reward}')

logits [not hate, hate]: [-0.6921191811561584, 0.37227320671081543]
probabilities [not hate, hate]: [0.2564709782600403, 0.7435290813446045]
reward (low): [-0.6921191811561584]


In [20]:
device_id = 0 if torch.cuda.is_available() else 'cpu'

sentiment_pipe = pipeline('sentiment-analysis',
                          model=toxicity_model_name,
                          device=device_id,
                          framework='pt')

reward_logits_kwargs = {
    'top_k': None, # Return all scores.
    'function_to_apply': 'none', # Set to 'none' to retrieve raw logits.
    'batch_size': 16
}

reward_probabilities_kwargs = {
    'top_k': None, # Return all scores.
    'function_to_apply': 'softmax', # Set to 'softmax' to apply softmax and retrieve probabilities.
    'batch_size': 16
}

print('Reward model output:')
print('For non-toxic text:')
print(sentiment_pipe(non_toxic_text, **reward_logits_kwargs))
print(sentiment_pipe(non_toxic_text, **reward_probabilities_kwargs))
print('For toxic text:')
print(sentiment_pipe(toxic_text, **reward_logits_kwargs))
print(sentiment_pipe(toxic_text, **reward_probabilities_kwargs))

Reward model output:
For non-toxic text:
[{'label': 'nothate', 'score': 3.114103078842163}, {'label': 'hate', 'score': -2.4896199703216553}]
[{'label': 'nothate', 'score': 0.9963293671607971}, {'label': 'hate', 'score': 0.0036705993115901947}]
For toxic text:
[{'label': 'hate', 'score': 0.37227320671081543}, {'label': 'nothate', 'score': -0.6921191811561584}]
[{'label': 'hate', 'score': 0.7435290217399597}, {'label': 'nothate', 'score': 0.2564709782600403}]


In [21]:
print(sentiment_pipe(non_toxic_text, **reward_logits_kwargs))
print(sentiment_pipe(non_toxic_text, **reward_probabilities_kwargs))

[{'label': 'nothate', 'score': 3.114103078842163}, {'label': 'hate', 'score': -2.4896199703216553}]
[{'label': 'nothate', 'score': 0.9963293671607971}, {'label': 'hate', 'score': 0.0036705993115901947}]


In [22]:
print(sentiment_pipe(toxic_text, **reward_logits_kwargs))
print(sentiment_pipe(toxic_text, **reward_probabilities_kwargs))

[{'label': 'hate', 'score': 0.37227320671081543}, {'label': 'nothate', 'score': -0.6921191811561584}]
[{'label': 'hate', 'score': 0.7435290217399597}, {'label': 'nothate', 'score': 0.2564709782600403}]


In [23]:
toxicity_evaluator = evaluate.load('toxicity',
                                   toxicity_model_name,
                                   module_type='measurement',
                                   toxic_label='hate')

In [24]:
non_toxic_text = "#Person 1# tells Tommy that he didn't like the movie."
toxicity_score = toxicity_evaluator.compute(predictions=[
    non_toxic_text
])

print('Toxicity score for non-toxic text:')
print(toxicity_score['toxicity'])

toxicity_score = toxicity_evaluator.compute(predictions=[
    toxic_text
])

print('Toxicity score for toxic text:')
print(toxicity_score['toxicity'])

Toxicity score for non-toxic text:
[0.0036705993115901947]
Toxicity score for toxic text:
[0.7435290217399597]


In [25]:
def evaluate_toxicity(model,
                      toxicity_evaluator,
                      tokenizer,
                      dataset,
                      num_samples):
    """
    Preprocess the dataset and split it into train and test parts.

    Parameters:
    - model (trl model): Model to be evaluated.
    - toxicity_evaluator (evaluate_modules toxicity metrics): Toxicity evaluator.
    - tokenizer (transformers tokenizer): Tokenizer to be used.
    - dataset (dataset): Input dataset for the evaluation.
    - num_samples (int): Maximum number of samples for the evaluation.

    Returns:
    tuple: A tuple containing two numpy.float64 values:
    - mean (numpy.float64): Mean of the samples toxicity.
    - std (numpy.float64): Standard deviation of the samples toxicity.
    """

    max_new_tokens = 100
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    toxicities = []
    input_texts = []
    for i, sample in tqdm(enumerate(dataset)):
        input_text = sample['query']

        if i > num_samples:
            break

        model = model.to(device)

        input_ids = tokenizer(input_text, return_tensors='pt', padding=True).input_ids.to(device)

        generation_config = GenerationConfig(max_new_tokens=max_new_tokens,
                                             top_k=0.0,
                                             top_p=1.0,
                                             do_sample=True)

        with torch.no_grad():
            response_token_ids = model.generate(
                input_ids=input_ids,
                generation_config=generation_config
            )

        # Ensure response is on CPU for decoding
        generated_text = tokenizer.decode(response_token_ids[0].cpu(), skip_special_tokens=True)

        toxicity_score = toxicity_evaluator.compute(predictions=[(input_text + ' ' + generated_text)])

        toxicities.extend(toxicity_score['toxicity'])

    # Compute mean and std using np.
    mean = np.mean(toxicities)
    std = np.std(toxicities)

    return mean, std

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)

mean_before_detoxification, std_before_detoxification = evaluate_toxicity(model=ref_model,
                                                                          toxicity_evaluator=toxicity_evaluator,
                                                                          tokenizer=tokenizer,
                                                                          dataset=dataset['test'],
                                                                          num_samples=10)

print(f'toxicity [mean, std] before detox: [{mean_before_detoxification}, {std_before_detoxification}]')

11it [01:48,  9.87s/it]

toxicity [mean, std] before detox: [0.0459878050182438, 0.05564359750859294]


In [27]:
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

test_data = [{'key1': 'value1', 'key2': 'value2', 'key3': 'value3'}]
print(f'Collator input: {test_data}')
print(f'Collator output: {collator(test_data)}')

Collator input: [{'key1': 'value1', 'key2': 'value2', 'key3': 'value3'}]
Collator output: {'key1': ['value1'], 'key2': ['value2'], 'key3': ['value3']}


In [28]:
learning_rate=1.41e-5
max_ppo_epochs=1
mini_batch_size=4
batch_size=16

# config = PPOConfig(
#     model_name=model_name,
#     learning_rate=learning_rate,
#     num_ppo_epochs=max_ppo_epochs,
#     mini_batch_size=mini_batch_size,
#     batch_size=batch_size,
#     bf16=False,
#     fp16=False,
#     use_cpu=True
# )

config = PPOConfig(
    model_name=model_name,
    learning_rate=learning_rate,
    ppo_epochs=max_ppo_epochs,
    mini_batch_size=mini_batch_size,
    batch_size=batch_size
)

ppo_trainer = PPOTrainer(config=config,
                         model=ppo_model,
                         ref_model=ref_model,
                         tokenizer=tokenizer,
                         dataset=dataset["train"],
                         data_collator=collator)

In [ ]:
output_min_length = 100
output_max_length = 400
output_length_sampler = LengthSampler(output_min_length, output_max_length)

generation_kwargs = {
    'min_length': 5,
    'top_k': 0.0,
    'top_p': 1.0,
    'do_sample': True
}

reward_kwargs = {
    'top_k': None, # Return all scores.
    'function_to_apply': 'none', # You want the raw logits without softmax.
    'batch_size': 16
}

max_ppo_steps = 10

for step, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    # Break when you reach max_steps.
    if step >= max_ppo_steps:
        break

    prompt_tensors = batch['input_ids']

    # Get response from FLAN-T5/PEFT LLM.
    summary_tensors = []

    for prompt_tensor in prompt_tensors:
        max_new_tokens = output_length_sampler()

        generation_kwargs['max_new_tokens'] = max_new_tokens
        summary = ppo_trainer.generate(prompt_tensor, **generation_kwargs)

        summary_tensors.append(summary.squeeze()[-max_new_tokens:])

    # This needs to be called "response".
    batch['response'] = [tokenizer.decode(r.squeeze()) for r in summary_tensors]

    # Compute reward outputs.
    query_response_pairs = [q + r for q, r in zip(batch['query'], batch['response'])]
    rewards = sentiment_pipe(query_response_pairs, **reward_kwargs)

    # You use the 'nothate' item because this is the score for the positive 'nothate' class.
